# Train DeepMind GNN (Custom Generated Dataset)
This notebook provisions a Kaggle GPU Environment, generates a complete N-Body gravitational physics dataset from scratch utilizing purely native Python `.npz` binaries, and then executes the GNN optimization pipeline against it natively.

In [ ]:
!rm -rf gnn-physics-simulator
!git clone https://github.com/MarkoMile/gnn-physics-simulator.git
%cd gnn-physics-simulator

# 1. Force downgrade PyTorch to 2.8.0 (latest version supported by PyG wheels)
!pip install torch==2.8.0+cu126 --extra-index-url https://download.pytorch.org/whl/cu126

# 2. Install PyTorch Geometric extensions using pre-built wheels for torch-2.8.0+cu126
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch-geometric

# Install utility dependencies (Weights & Biases, PyYAML, TF Data Parser)
!pip install wandb pyyaml tensorflow

In [ ]:
import tqdm.notebook
import tqdm.auto

# Force the auto-detector to use the notebook progressive version to avoid console flooding
tqdm.auto.tqdm = tqdm.notebook.tqdm
tqdm.auto.trange = tqdm.notebook.trange

In [ ]:
# Generate a custom local dataset utilizing numerical integration
# (Utilizing default bounds in config.yaml -> 'simulation' section)
!python scripts/generate_dataset.py --config configs/config.yaml --output data/GeneratedNBody

import yaml

# Patch the config.yaml dynamically to target the generated synthetic dataset instead of the tiny sample
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
    
config['data']['dataset_path'] = 'data/GeneratedNBody'
config['data']['dataset_format'] = 'npz'

with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f)
    
print("Patched config.yaml successfully to target full data/GeneratedNBody!")

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    
    wandb.login()
    print("WandB logged in successfully using Kaggle Secrets.")
except Exception as e:
    print(f"WandB login skipped/failed: {e}\nEnsure you have added 'wandb_api_key' to your Kaggle Secrets if you want telemetry.")

In [ ]:
# Execute the global training loop with WandB telemetry enabled!
!python train.py --config configs/config.yaml --wandb